# 00_core — shared constants and the shell helper

**Purpose.** Every later notebook depends on the same five facts: the
repository path, the namespace, the two container image tags, the
protected edge URL, and the unprotected victim URL. Defining them once
here keeps the rest of the lab referencing one set of names instead of
repeated string literals.

**Execution.** Cells top-to-bottom; the narration above each cell states
what it does and what its output should look like. Only the final test
cell touches the cluster — that test is the lab's front door, so a
failure there blocks everything downstream until connectivity is
resolved. Prerequisites: the NixOS VM is converged and `nix develop` is
active ([INSTALL.md](../INSTALL.md)); `kubectl get nodes` alone should
already list one Ready node.
**Execution context.** Jupyter runs **inside the VM**
(`orb -m aisec-lab` → `nix develop` → `jupyter notebook`), browsed from
the Mac at `http://localhost:8888` — OrbStack forwards VM ports to Mac
localhost automatically. Cells execute where the toolchain and
KUBECONFIG live; outputs render in the Mac browser.

**Run order for the whole lab:** [00_core](00_core.ipynb) (this) →
[01_nova_rules](01_nova_rules.ipynb) → [02_gate_app](02_gate_app.ipynb) →
[03_victim](03_victim.ipynb) → [04_deploy](04_deploy.ipynb) →
[05_attacks](05_attacks.ipynb). Each notebook ends by naming the next.


In [ ]:
#| default_exp core
#| export
from pathlib import Path
import json, os, subprocess

# Where the lab lives. nbdev_test always runs from the repo root, so the
# default is simply "here"; the env var is the escape hatch for ad-hoc runs.
LAB = Path(os.environ.get("AI_SEC_LAB", Path.cwd()))   # run from the repo root — nbdev_test does

NS           = "ai-sec"                           # every object lands in this namespace
GATE_IMAGE   = "ai-sec-lab/laya-gate:1.0.0"       # built from gate/Dockerfile (02_gate_app)
VICTIM_IMAGE = "ai-sec-lab/atlas-victim:1.0.0"    # built from gate/victim_Dockerfile (03_victim)
EDGE_URL     = "http://ai-sec.lab.internal"       # PROTECTED path: Gateway → gate (04_deploy)
VICTIM_SVC   = "http://atlas.ai-sec.svc.cluster.local:8080"   # UNPROTECTED path, cluster-internal only


**Cell function.** Five constants. The asymmetry between them is the
spine of the lab: `EDGE_URL` is what a real client reaches — Gateway,
then gate, then app. `VICTIM_SVC` is the same app reached from *inside*
the cluster, past the gate — the path 05_attacks uses to prove the
vulnerability is real before proving the gate blocks it.
**Next cell:** the `sh()` helper every later test executes through.

In [ ]:
#| export
def sh(cmd: str, **kw) -> subprocess.CompletedProcess:
    """Run a shell command in the lab dir; raise on failure (fail closed)."""
    r = subprocess.run(cmd, shell=True, cwd=LAB, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"[{cmd}] failed:\n{r.stderr}")
    return r


**Why fail-closed on a shell helper.** Every later notebook drives
`kubectl`, `tofu`, and `curl` through `sh()`. A non-zero exit raises
immediately, so the lab stops loudly instead of continuing
half-configured — a security lab that kept running after a failed
command would itself be a fail-open system.
**Next cell:** the front-door connectivity test.

In [ ]:
# test: cluster reachable
r = sh("kubectl get nodes -o json")
nodes = json.loads(r.stdout)["items"]
assert len(nodes) == 1 and nodes[0]["status"]["conditions"][-1]["type"] == "Ready"
print("cluster OK:", nodes[0]["metadata"]["name"])


**Expected output:** `cluster OK: aisec-lab` (the VM's hostname).

**On failure** with `RuntimeError: [kubectl get nodes…]`: the cluster is
not up — converge the machine again (`nixos-rebuild switch --flake
.#aisec-lab`, [INSTALL.md](../INSTALL.md) §Part 2), confirm `kubectl get
nodes` manually, then re-run this cell. Every later test assumes this
one passes.

**Next notebook:** [01_nova_rules.ipynb](01_nova_rules.ipynb) — the
policy, as tested code.